# 01 — Lazy Evaluation & the Spark DAG

**Concept:** Spark builds a logical plan (DAG) as you chain transformations, but executes *nothing* until you call an action.  
**Why it matters for interviews:** Every question about performance, shuffles, or caching starts here.

## What you will observe
1. Transformations are free — they just extend the DAG.
2. Actions trigger a job; you can see exactly what Spark plans via `.explain()`.
3. Wide transformations (shuffles) introduce stage boundaries — visible in the plan.
4. Classic interview gotchas: re-triggering jobs accidentally.

In [1]:
import sys
import os
from pathlib import Path
import time

sys.path.append(str(Path().absolute().parent))

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("01-lazy-evaluation")
    .config("spark.sql.adaptive.enabled", "true")
    # Arrow-based pandas↔Spark transfer: createDataFrame(pandas_df) sends data
    # to the JVM as an Arrow IPC buffer — no Python worker processes needed.
    # Without this, Spark falls back to applySchemaToPythonRDD (Python pickle),
    # which spawns workers and crashes on Windows + Spark 4.x.
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.python.worker.faulthandler.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — UI: http://localhost:4040")
print(f"Worker Python: {sys.executable}")
print(f"Arrow enabled: {spark.conf.get('spark.sql.execution.arrow.pyspark.enabled')}")

# spark.range(5).show()
# print("smoke test passed")

Spark 4.1.1 — UI: http://localhost:4040
Worker Python: c:\Users\krivg\spark-bq\.venv\Scripts\python.exe
Arrow enabled: true


In [ ]:
# Default number of shuffle partitions is 200, which is excessive for local testing.
spark.conf.get("spark.sql.shuffle.partitions")

'200'

## 1. Build sample data

We use a hard-coded list so there is no I/O noise — the focus is on the execution model, not data sources.

In [2]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Keep rows as a Python list — reused in section 6 (JSON schema-inference demo)
rows = [
    (1,  "alice",  "eng",    95000.0),
    (2,  "bob",    "sales",  72000.0),
    (3,  "carol",  "eng",   105000.0),
    (4,  "dave",   "sales",  68000.0),
    (5,  "eve",    "eng",    88000.0),
    (6,  "frank",  "hr",     61000.0),
    (7,  "grace",  "hr",     63000.0),
    (8,  "heidi",  "eng",   112000.0),
    (9,  "ivan",   "sales",  74000.0),
    (10, "judy",   "eng",    99000.0),
]

schema = StructType([
    StructField("id",     IntegerType(), False),
    StructField("name",   StringType(),  True),
    StructField("dept",   StringType(),  True),
    StructField("salary", DoubleType(),  True),
])

pdf = pd.DataFrame(rows, columns=["id", "name", "dept", "salary"])

# With spark.sql.execution.arrow.pyspark.enabled=true, Spark transfers the
# pandas DataFrame to the JVM as an Arrow IPC buffer — no Python worker processes.
# The physical plan will show LocalTableScan instead of applySchemaToPythonRDD.
df = spark.createDataFrame(pdf, schema=schema)

print("df created — check plan to confirm Arrow path (no applySchemaToPythonRDD):")
df.explain()
print(f"Partitions: {df.rdd.getNumPartitions()}")

df created — check plan to confirm Arrow path (no applySchemaToPythonRDD):
== Physical Plan ==
LocalTableScan [id#0, name#1, dept#2, salary#3]


Partitions: 8


## 2. Chaining transformations — building the DAG

Each call below returns a *new* DataFrame object. Nothing runs. Spark just appends a node to the logical plan.

In [3]:
# NARROW transformation: filter does not require data movement between partitions.
step1 = df.filter(F.col("salary") > 70000)

# NARROW: withColumn adds a computed column per-row, same partition.
step2 = step1.withColumn("salary_k", F.col("salary") / 1000)

# WIDE transformation: groupBy requires a shuffle — all rows for the same dept
# key must land on the same partition. This is where a stage boundary appears.
step3 = step2.groupBy("dept").agg(
    F.count("*").alias("headcount"), # count("*") -> count rows, not just non-null salaries
    F.avg("salary_k").alias("avg_salary_k"),
)

# NARROW: orderBy on the result (technically triggers another shuffle for global sort).
result = step3.orderBy("avg_salary_k", ascending=False)

print("Four transformation steps defined.")
print("Check the Spark UI Jobs tab — zero jobs should have run so far.")

Four transformation steps defined.
Check the Spark UI Jobs tab — zero jobs should have run so far.


## 3. Inspect the plan with `.explain()` — *before* any action

`.explain()` itself is not an action (it does not compute data). It prints the optimized physical plan Catalyst produced from the logical plan you built above.

**What to look for:**
- `Exchange` nodes = shuffle = stage boundary
- `Project` / `Filter` = narrow, pushed down as far as possible
- `HashAggregate` (partial) → `Exchange` → `HashAggregate` (final) = the classic 2-phase aggregation pattern

#### Parsed Logical plan is a unresolved plan that extracted from the query
#### Analyzed logical plans transforms which translates unresolvedAttribute and unresolvedRelation into fully typed objects
#### The optimized logical plan transforms through a set of optimization rules, resulting in the physical plan.

In [4]:
print("=" * 60)
print("SIMPLE (default) plan:")
print("=" * 60)
result.explain()

print("\n" + "=" * 60)
print("EXTENDED plan — logical + optimized + physical:")
print("=" * 60)
result.explain(mode="extended")

SIMPLE (default) plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [avg_salary_k#6 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(avg_salary_k#6 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=21]
      +- HashAggregate(keys=[dept#2], functions=[count(1), avg(salary_k#4)])
         +- Exchange hashpartitioning(dept#2, 200), ENSURE_REQUIREMENTS, [plan_id=18]
            +- HashAggregate(keys=[dept#2], functions=[partial_count(1), partial_avg(salary_k#4)])
               +- LocalTableScan [dept#2, salary_k#4]



EXTENDED plan — logical + optimized + physical:
== Parsed Logical Plan ==
'Sort ['avg_salary_k DESC NULLS LAST], true
+- Aggregate [dept#2], [dept#2, count(1) AS headcount#5L, avg(salary_k#4) AS avg_salary_k#6]
   +- Project [id#0, name#1, dept#2, salary#3, (salary#3 / cast(1000 as double)) AS salary_k#4]
      +- Filter (salary#3 > cast(70000 as double))
         +- LocalRelation [id#0, name#1, dept#2, salary#3]

== Analyzed Logical Plan ==
d

In [5]:
# FORMATTED mode is most readable for stage-boundary analysis
print("FORMATTED plan — tree with operators and metrics:")
result.explain(mode="formatted")

FORMATTED plan — tree with operators and metrics:
== Physical Plan ==
AdaptiveSparkPlan (7)
+- Sort (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Exchange (3)
            +- HashAggregate (2)
               +- LocalTableScan (1)


(1) LocalTableScan
Output [2]: [dept#2, salary_k#4]
Arguments: [dept#2, salary_k#4]

(2) HashAggregate
Input [2]: [dept#2, salary_k#4]
Keys [1]: [dept#2]
Functions [2]: [partial_count(1), partial_avg(salary_k#4)]
Aggregate Attributes [3]: [count#14L, sum#15, count#16L]
Results [4]: [dept#2, count#17L, sum#18, count#19L]

(3) Exchange
Input [4]: [dept#2, count#17L, sum#18, count#19L]
Arguments: hashpartitioning(dept#2, 200), ENSURE_REQUIREMENTS, [plan_id=18]

(4) HashAggregate
Input [4]: [dept#2, count#17L, sum#18, count#19L]
Keys [1]: [dept#2]
Functions [2]: [count(1), avg(salary_k#4)]
Aggregate Attributes [2]: [count(1)#12L, avg(salary_k#4)#13]
Results [3]: [dept#2, count(1)#12L AS headcount#5L, avg(salary_k#4)#13 AS avg_salary_k#6]

(5) Exch

## 4. Actions — triggering execution

Only these calls cause Spark to submit a job:

| Action | Notes |
|--------|-------|
| `count()` | Returns a Python int — full scan |
| `show()` / `display()` | Fetches rows to driver |
| `collect()` | Pulls *all* rows to driver — dangerous on large data |
| `write.*` | Materializes to storage |
| `first()` / `take(n)` | Partial scan, but still a job |
| `toPandas()` | Equivalent to collect() |

Watch the Spark UI Jobs tab after each cell below.

In [6]:
# ACTION 1: show() — triggers Job 0 (or next job number)
t0 = time.perf_counter()
result.show()
t1 = time.perf_counter()
print(f"show() took {t1 - t0:.2f}s — check Spark UI for stage breakdown")


# After the action, AQE has finalized the plan — explain now says isFinalPlan=true
# and shows the ACTUAL operators and stage merges AQE applied at runtime.
# Compare this to the pre-execution plan from section 3 to see what AQE changed.
# print("\nFinal plan (post-execution, AQE decisions visible):")
# result.explain(mode="formatted")

+-----+---------+------------+
| dept|headcount|avg_salary_k|
+-----+---------+------------+
|  eng|        5|        99.8|
|sales|        2|        73.0|
+-----+---------+------------+

show() took 2.34s — check Spark UI for stage breakdown


## Why are stages skipped?
You have AQE enabled (spark.sql.adaptive.enabled=true). With AQE, Spark can look at the actual shuffle output size from Stage N before deciding how to run Stage N+1. For your 10-row dataset, AQE sees the shuffled data is tiny and collapses or eliminates subsequent stages entirely.

## The specific rule that fires: 
AQE coalesces shuffle partitions. The default is 200 shuffle partitions (spark.sql.shuffle.partitions=200). After stage 0 writes 200 shuffle files, AQE reads the file sizes, sees most are empty, and coalesces them into 1 or 2 real partitions — sometimes collapsing what would have been two separate stages into one. Skipped stages = stages Spark planned but AQE proved were unnecessary at runtime.

You can watch AQE's decisions change the plan by adding this cell after any action:

## WholeStageCodegen 
Is the wrapper you'll see around most of these — it's Tungsten fusing multiple operators into a single JVM bytecode loop. One WholeStageCodegen box in the UI = multiple logical operators compiled into one tight loop with no per-row object allocation

In [ ]:
# ACTION 2: count() — triggers a SECOND job even though same DataFrame.
# Without caching, Spark re-executes the entire plan from scratch.
t0 = time.perf_counter()
n = result.count()
t1 = time.perf_counter()
print(f"count() = {n}, took {t1 - t0:.2f}s")
print("Spark UI should now show TWO completed jobs — same data read twice.")

## 5. Gotcha — calling count() in a loop

A classic anti-pattern: each `count()` inside a loop re-executes the full plan.  
**Interview answer:** Cache the DataFrame before the loop, or redesign to a single aggregation.

In [ ]:
depts = ["eng", "sales", "hr"]

print("BAD — N separate jobs:")
t0 = time.perf_counter()
for dept in depts:
    # Each iteration submits a new job — filter + scan repeated N times
    n = df.filter(F.col("dept") == dept).count()
    print(f"  {dept}: {n} employees")
print(f"Elapsed: {time.perf_counter() - t0:.2f}s")

print()
print("GOOD — single job, all depts in one pass:")
t0 = time.perf_counter()
dept_counts = (
    df.groupBy("dept")
    .count()
    .collect()  # one job
)
for row in dept_counts:
    print(f"  {row['dept']}: {row['count']} employees")
print(f"Elapsed: {time.perf_counter() - t0:.2f}s")

## 6. Gotcha — schema inference triggers a job

When reading from JSON/CSV without an explicit schema, Spark runs a **sampling job** to infer types.  
This is a hidden action — it happens at `spark.read.json(path)` before you call any explicit action.  
**Always provide explicit schemas in production code.**

In [ ]:
import json, tempfile, os

# Write a tiny JSON file just to demonstrate the point
tmp = tempfile.mkdtemp()
json_path = os.path.join(tmp, "sample.json")
with open(json_path, "w") as fh:
    for r in rows:
        fh.write(json.dumps({"id": r[0], "name": r[1], "dept": r[2], "salary": r[3]}) + "\n")

print("Without schema — Spark runs a hidden inference job:")
t0 = time.perf_counter()
df_infer = spark.read.json(json_path)  # job fires HERE
print(f"  read.json (no schema) returned in {time.perf_counter() - t0:.3f}s — a job ran")
print(f"  Inferred schema: {df_infer.schema.simpleString()}")

print()
print("With explicit schema — truly lazy, no job at read time:")
t0 = time.perf_counter()
df_explicit = spark.read.schema(schema).json(json_path)  # no job
print(f"  read.json (with schema) returned in {time.perf_counter() - t0:.4f}s — no job ran")

## 7. Stage boundaries — seeing shuffles in the plan

Every `Exchange` in the physical plan = a shuffle = a stage boundary.  
Stages within a job run sequentially. Tasks within a stage run in parallel.

```
Job
 └── Stage 0: scan → filter → partial-agg      (narrow, parallelizable)
      [shuffle write]
 └── Stage 1: shuffle-read → final-agg → sort   (starts after Stage 0)
```

In [ ]:
# Narrow-only pipeline: no Exchange nodes expected
narrow_only = df.filter(F.col("salary") > 80000).withColumn("bonus", F.col("salary") * 0.1)
print("Narrow-only plan (expect NO Exchange):")
narrow_only.explain()

print()

# Wide pipeline: groupBy forces shuffle — Exchange appears
wide = df.groupBy("dept").agg(F.sum("salary").alias("total_salary"))
print("Wide plan (expect Exchange before HashAggregate final):")
wide.explain()

## 8. Caching — breaking the re-execution cycle

`.cache()` / `.persist()` tell Spark to **store the result** of a plan in memory (or disk) after the first action materializes it. Subsequent actions on the same DataFrame skip re-computation.

Rule of thumb: cache when a DataFrame is used **more than once** downstream.

In [ ]:
# Cache after a wide transformation that is reused
dept_summary = (
    df.groupBy("dept")
    .agg(
        F.count("*").alias("headcount"),
        F.avg("salary").alias("avg_salary"),
    )
)

# LAZY: .cache() marks the DataFrame for caching but does NOT trigger execution.
dept_summary.cache()
print(".cache() called — still no job yet (lazy marking)")

# FIRST action: materializes and fills the cache
t0 = time.perf_counter()
dept_summary.show()
t1 = time.perf_counter()
print(f"First show(): {t1 - t0:.3f}s — computed AND cached")

# SECOND action: reads from cache, no recomputation
t0 = time.perf_counter()
n = dept_summary.count()
t1 = time.perf_counter()
print(f"count() after cache: {t1 - t0:.3f}s (should be noticeably faster)")

# Always unpersist when done — Spark won't evict proactively until memory pressure
dept_summary.unpersist()
print("Unpersisted — memory released.")

## 9. Key takeaways — interview cheat sheet

| Concept | One-liner |
|---------|----------|
| Lazy evaluation | Transformations build a DAG; actions submit a job |
| `.explain()` | Reads the plan without triggering a job — safe to call anytime |
| `Exchange` in plan | A shuffle — introduces a stage boundary |
| Stage | Set of tasks that can run in parallel without a shuffle |
| Re-execution | Without cache, every action re-runs the full plan |
| Schema inference | `read.json/csv` without schema fires a hidden sampling job |
| Cache timing | `.cache()` is lazy; the first action materializes it |
| Unpersist | Always call `.unpersist()` when the cached DF is no longer needed |

**Common interview question:** *"What is the difference between a transformation and an action?"*  
Answer: Transformations are lazy — they extend the logical plan and return a new DataFrame. Actions are eager — they trigger a Catalyst optimization pass, produce a physical plan, and submit one or more jobs to the cluster.

**Next notebook:** `02_shuffling_and_stages.ipynb` — wide vs. narrow in depth, partition counts, skew.

In [ ]:
# Spark session stays alive for the UI — call spark.stop() when truly done
print("Notebook complete. Spark UI still available at http://localhost:4040")
print("Run spark.stop() in a new cell to shut down.")